# Tokenization

## Goal
Inspect how source and target text become sequence-to-sequence model inputs.

## Setup
Select **Python (machine-translation-lab)** or the project `.venv` kernel. All code cells are intentionally TODO-only. Work through the [Hugging Face translation tutorial](https://huggingface.co/docs/transformers/tasks/translation). No training, data loading, or model loading happens automatically.

In [1]:
# TODO: Add your imports and confirm the selected Python interpreter.

from pathlib import Path
from datasets import load_dataset



In [2]:
# TODO: Load the dataset when you are ready. Inspect available splits and features.

books = load_dataset(
    "Helsinki-NLP/opus_books",
    "en-fr",
    cache_dir=str(Path.home() / ".cache" / "huggingface" / "datasets"),
)

## Steps

### 1. Tokenizer loading
Use `AutoTokenizer` with `google-t5/t5-small`. This downloads small tokenizer assets when you implement the cell.

In [3]:
# TODO: Load the tokenizer and inspect its special tokens. 

from transformers import AutoTokenizer

checkpoint = "google-t5/t5-small"
tokenizer = AutoTokenizer.from_pretrained(checkpoint) 

print(tokenizer.special_tokens_map)

{'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}


### 2. Source/target tokenization
Use the T5 task prefix `translate English to French: `. The current API supports `text_target` for target text.

In [4]:
# TODO: Tokenize your own source/target pairs and inspect the returned fields.

source_lang = "en"
target_lang = "fr"
prefix = "translate English to French: "


def preprocess_function(examples):
    inputs = [prefix + example[source_lang] for example in examples["translation"]]
    targets = [example[target_lang] for example in examples["translation"]]
    model_inputs = tokenizer(inputs, text_target=targets, max_length=128, truncation=True)
    return model_inputs 

In [5]:
tokenized_books = books.map(preprocess_function, batched=True)

### 3. Padding and truncation
Compare fixed padding with dynamic padding. `DataCollatorForSeq2Seq` supports paired batches; provide a model object or omit the model argument, rather than a checkpoint string.

In [6]:
# TODO: Experiment with padding and truncation on sentences of different lengths.

from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer) 

In [7]:
features = [
    {key: row[key] for key in ("input_ids", "attention_mask", "labels")}
    for row in tokenized_books["train"].select(range(4))
]
batch = data_collator(features)
print({key: value.shape for key, value in batch.items()})

{'input_ids': torch.Size([4, 13]), 'attention_mask': torch.Size([4, 13]), 'labels': torch.Size([4, 8])}


### 4. Understanding model inputs
Examine `input_ids`, `attention_mask`, and `labels`. Investigate padding, special tokens, and the ignored-label value -100.

In [8]:
# See the batch dimensions
for name, values in batch.items():
    print(name, values.shape)

# Inspect the first English example
print("Token IDs:", batch["input_ids"][0])
print("Attention mask:", batch["attention_mask"][0])

# Convert its IDs back into readable text
print("English:", tokenizer.decode(
    batch["input_ids"][0], skip_special_tokens=True
))

# Remove ignored padding positions before decoding the French target
label_ids = batch["labels"][0]
label_ids = label_ids[label_ids != -100]

print("French:", tokenizer.decode(
    label_ids, skip_special_tokens=True
))

input_ids torch.Size([4, 13])
attention_mask torch.Size([4, 13])
labels torch.Size([4, 8])
Token IDs: tensor([13959,  1566,    12,  2379,    10,    37, 12832,    49,     1,     0,
            0,     0,     0])
Attention mask: tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
English: translate English to French: The Wanderer
French: Le grand Meaulnes


## Checks
Restart the kernel and run your completed cells in order. Record what you checked, any assumptions, and remaining questions.

In [9]:
# TODO: Add your checks and observations.

# TODO: Inspect shapes and decode tokens to explain what the model receives. 

# Each input token has a matching attention-mask entry
assert batch["input_ids"].shape == batch["attention_mask"].shape

# Each English example has a French target
assert batch["input_ids"].shape[0] == batch["labels"].shape[0]

# Your preprocessing limits both sequences to 128 tokens
assert batch["input_ids"].shape[1] <= 128
assert batch["labels"].shape[1] <= 128

# Input positions marked as padding contain the padding token
padding_positions = batch["attention_mask"] == 0
assert (batch["input_ids"][padding_positions] == tokenizer.pad_token_id).all()

print("All checks passed!")

All checks passed!


## Next Steps
Save your observations and continue to the next notebook. After notebook 4, review the complete pipeline. Fine-tuning is optional later and is outside this setup.